Live Deployment Link: <https://phones-price-prediction-areeba-tahir.streamlit.app/>

GitHub Repository: <https://github.com/areebaa063/phones-price-prediction>

## Setup: Running This Notebook on Google Colab

No installation is needed. Google Colab already has every library this notebook uses (XGBoost is pre-installed on Colab).

1. Go to **https://colab.research.google.com** and sign in with a Google account.
2. Click **File → Upload notebook** and choose this `.ipynb` file.
3. Click **Runtime → Run all**.
4. When **Step 2.1** runs, a **"Choose Files"** button appears below that cell. Click it and select **`cleaned_all_phones.csv`** from your computer. The notebook continues automatically after the upload.

> **Note:** Colab deletes uploaded files when the runtime is reset or disconnected. If you get a *"file not found"* error, run Step 2.1 again and re-upload the CSV.

---

## Introduction

##### **Definition** A regression ML task that predicts the price (USD) of a smartphone from its hardware specifications, operating system, brand and release date.
##### **Application** Retail and e-commerce price suggestions, second-hand phone valuation, trade-in offers, market analysis and consumer purchasing guidance.

### Aim of This Notebook

> **Which of 5 regression algorithms predicts smartphone prices best, and how sure can we be about that answer?**

To answer it fairly we use **K-Fold Cross Validation** and compare the algorithms using **Root Mean Squared Error (RMSE)**.

This notebook uses the **full dataset (1,512 phones, 1,510 after cleaning)** with **10-fold cross validation**.

### What is K-Fold Cross Validation?

A single train/test split can be lucky or unlucky: the result depends on *which* phones happened to land in the test set. K-Fold Cross Validation removes this luck:

1. Shuffle the data and split it into **K equal parts (folds)**.
2. Train the model on **K−1 folds** and test it on the **remaining fold**.
3. Repeat **K times**, so that **every fold is used as the test set exactly once**.
4. Report the **average** score over the K rounds (and the standard deviation, to show how much it varies).

With **K = 10** on **1,510 phones**, each test fold has 151 phones and each training set has 1,359 phones:

```
Round  1:  [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
Round  2:  [train] | [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
Round  3:  [train] | [train] | [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
  ...
Round 10:  [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [TEST ]
```

Every phone is used for testing **exactly once** and for training **K−1 times**.

### Evaluation Measure: Root Mean Squared Error (RMSE)

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

- $y_i$ = actual price, $\hat{y}_i$ = predicted price, $n$ = number of test phones
- RMSE is in the **same unit as the output (US dollars)**, so it reads as *"on average, predictions are off by about $X"*.
- **Lower RMSE = better model.** Because errors are squared, RMSE penalises **large mistakes** heavily.

## Input and Output

### Input (Features / Attributes)

The raw CSV has 20 specification columns (plus the phone name and the price). In **Step 4** they are converted into **22 model inputs**: the resolution text becomes a width and a height, the announcement date becomes a release year, and the OS text is split into an OS family and an OS version.

| Feature | Description | Type | Possible Values |
|---|---|---|---|
| **inches** | Screen size (inches) | Numerical | 3.8 – 10.4 |
| **battery** | Battery capacity (mAh) | Numerical | 1,821 – 7,250 |
| **ram(GB)** | Memory (GB) | Numerical | 1 – 24 |
| **weight(g)** | Weight (grams) | Numerical | 130 – 500 |
| **storage(GB)** | Internal storage (GB) | Numerical | 1 – 512 |
| **res_width** | Screen resolution width in pixels (from `resolution`) | Numerical | 480 – 3,840 |
| **res_height** | Screen resolution height in pixels (from `resolution`) | Numerical | 800 – 3,840 |
| **release_year** | Announcement year + month/12 (from `announcement_date`) | Numerical | 2016.67 – 2023.58 |
| **os_version** | Version number of the operating system (from `os`) | Numerical | 3 – 16 |
| **video_720p … video_960fps** (10 columns) | Whether the camera supports that video resolution / frame rate (0 = no, 1 = yes) | Numerical (0/1) | 0, 1 |
| **brand** | Phone manufacturer | Categorical | Apple, Google, Honor, Huawei, Lenovo, LG, OnePlus, Oppo, Realme, Samsung, Sony, Vivo, Xiaomi |
| **os_family** | Operating system family (from `os`) | Categorical | Android, iOS, EMUI, Tizen |
| **battery_type** | Battery chemistry | Categorical | Li-Po, Li-Ion |

The 10 video columns are: `video_720p`, `video_1080p`, `video_4K`, `video_8K`, `video_30fps`, `video_60fps`, `video_120fps`, `video_240fps`, `video_480fps`, `video_960fps`.

The `phone_name` column is **not** used as an input: it is an identifier with almost one different value per phone, so it would only let a model memorise phones instead of learning from specifications.

### Output (Prediction)

| Output Attribute | Description | Data Type |
|---|---|---|
| **price(USD)** | Price of the phone in USD | Numerical |

### Example Records

| phone_name | brand | os | inches | resolution | battery | ram(GB) | storage(GB) | price(USD) |
|---|---|---|---|---|---|---|---|---|
| Y6II Compact | Huawei | Android 5.1 | 5.0 | 720x1280 | 2200 | 2 | 16 | 120 |
| K20 plus | LG | Android 7.0 | 5.3 | 720x1280 | 2700 | 2 | 16 | 100 |
| Redmi Note 4 | Xiaomi | Android 6.0 | 5.5 | 1080x1920 | 4100 | 4 | 32 | 150 |
| P10 | Huawei | Android 7.0 | 5.1 | 1080x1920 | 3200 | 4 | 32 | 420 |

## Developer & System Information

| **Attribute** | **Details** |
|---|---|
| **Developer Name** | Areeba Tahir (FA24-BED-014), Dr. Rao Muhammad Adeel Nawab |
| **Program Name** | `phones_price_prediction_10fold` |
| **IDE** | Google Colab |
| **Programming Language** | Python 3 |
| **Libraries** | NumPy, Pandas, scikit-learn, XGBoost, Matplotlib, tqdm, Pickle (built-in). Exact versions are printed in Step 1. |
| **Dataset** | `cleaned_all_phones.csv` (1,512 phones, 20 specification columns + phone name + price). A cleaned smartphone-specifications dataset covering 13 brands, announced between September 2016 and August 2023. |
| **GitHub** | https://github.com/areebaa063 |

### Table of Contents
- **Step 1:** Import Libraries
- **Step 2:** Load Sample Data
  - **Step 2.1:** Upload the Dataset (Google Colab)
  - **Step 2.2:** Read the Dataset
- **Step 3:** Understand and Pre-process Sample Data
  - **Step 3.1:** Understand Sample Data
  - **Step 3.2:** Pre-process Sample Data
- **Step 4:** Feature Extraction
- **Step 5:** Build the Pre-processing Pipeline
- **Step 6:** Define the 5 Regression Algorithms
- **Step 7:** Set Up 10-Fold Cross Validation
  - **Step 7.1:** Create the K-Fold Splitter
  - **Step 7.2:** Inspect the Folds
- **Step 8:** Run 10-Fold Cross Validation for All 5 Algorithms
- **Step 9:** Compare the Algorithms
  - **Step 9.1:** RMSE of Each Algorithm on Each Fold
  - **Step 9.2:** Summary Comparison Table
  - **Step 9.3:** Visual Comparison
  - **Step 9.4:** Observations
- **Step 10:** Train the Final Model
  - **Step 10.1:** Select the Best Algorithm
  - **Step 10.2:** Train the Best Algorithm on All the Data
  - **Step 10.3:** Save the Trained Model
- **Step 11:** Execute the Application Phase
  - **Step 11.1:** Take Input from User
  - **Step 11.2:** Convert User Input into Feature Vector
  - **Step 11.3:** Load the Saved Model
  - **Step 11.4:** Model Prediction
- **Step 12:** Execute the Feedback Phase
- **Conclusion**

# Code — Smartphone Price Prediction System
## Step 1: Import Libraries

In [6]:
# --- Core Python Libraries ---
import os
import pickle
import time
import warnings

import numpy as np
import pandas as pd

# --- Machine Learning (scikit-learn) ---
import sklearn
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# --- The 5 Regression Algorithms ---
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor
from xgboost import XGBRegressor

# --- Visualization ---
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator

# --- Progress Bar ---
from tqdm.auto import tqdm

# Keep the output clean (Lasso may print convergence notices on some versions)
warnings.filterwarnings("ignore")

# Print library versions for reproducibility
import xgboost
print(f"scikit-learn : {sklearn.__version__}")
print(f"xgboost      : {xgboost.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")

ModuleNotFoundError: No module named 'sklearn'

# Step 2: Load Sample Data
## Step 2.1: Upload the Dataset (Google Colab)

When this cell runs in Google Colab, a **"Choose Files"** button appears. Select **`cleaned_all_phones.csv`**.

If the file is already present (for example, you uploaded it earlier, or you are running the notebook on your own computer), the upload is skipped.

In [ ]:
DATA_FILE = "cleaned_all_phones.csv"

# google.colab is only available inside Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(DATA_FILE):
    print(f"Please click 'Choose Files' and select '{DATA_FILE}' ...")
    uploaded = files.upload()

    # If the browser renamed the file (e.g. 'cleaned_all_phones (1).csv'), rename it back
    uploaded_name = list(uploaded.keys())[0]
    if uploaded_name != DATA_FILE:
        os.rename(uploaded_name, DATA_FILE)

if os.path.exists(DATA_FILE):
    print(f"'{DATA_FILE}' is ready to use.")
else:
    raise FileNotFoundError(f"'{DATA_FILE}' not found. Please run this cell again and upload it.")

## Step 2.2: Read the Dataset

This notebook uses **all** the phones in the dataset.

In [ ]:
# Load the full phones dataset
phones = pd.read_csv(DATA_FILE)

print(f"Loaded dataset: {phones.shape[0]:,} phones, {phones.shape[1]} columns")

# Preview the dataset
phones.head()

# Step 3: Understand and Pre-process Sample Data
## Step 3.1: Understand Sample Data

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code helps us understand the dataset by:
1. Printing the names and data types of all attributes (columns).
2. Showing the total number of instances (rows).
3. Showing how many values are missing in each attribute.
"""

print("\nAttributes in Dataset:")
print("======================\n")
print(phones.dtypes.to_string())

print("\nNumber of Instances in Dataset:")
print("================================\n")
print(len(phones))

print("\nMissing Values in Each Attribute:")
print("==================================\n")
missing = phones.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

### Understanding the Output (price)

Before choosing an evaluation measure, it is important to look at the **range of the output**. Phone prices in this dataset are **right-skewed**: half of the phones cost $260 or less, and 95% cost under about $880, but a few flagship phones cost up to $2,300.

In [ ]:
# Summary statistics of the phone price
price_summary = phones["price(USD)"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
price_summary.to_frame(name="price ($)")

> **Why this matters for RMSE:** RMSE squares each error, so **one badly predicted flagship phone can raise the RMSE more than dozens of well-predicted budget phones**. Keep this in mind when reading the results: a large RMSE does not necessarily mean the model is bad for *typical* phones.

## Step 3.2: Pre-process Sample Data

In [ ]:
"""
Pre-processing steps:
---------------------
1. Drop 'phone_name'. It is an identifier (almost every phone has a different
   name), not a specification, so it cannot help predict the price of a new phone.
2. Remove duplicate rows (phones with identical specifications and price).
   Duplicates are harmful in cross validation because a phone could appear in
   both the training and test folds, making the RMSE look artificially optimistic.
3. Check that every price and every numeric specification is greater than 0.
4. Remaining missing values (if any) are filled inside the pipeline in Step 5
   using only training folds.
"""

# 1. Drop the identifier column
phones = phones.drop(columns=["phone_name"])
print("Dropped 'phone_name' identifier column.")

# 2. Drop duplicate rows
rows_before = len(phones)
phones = phones.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {rows_before - len(phones):,}")

# 3. Sanity check: no zero or negative values in the numeric specifications / price
numeric_columns_to_check = ["inches", "battery", "ram(GB)", "weight(g)", "storage(GB)", "price(USD)"]
invalid_rows = (phones[numeric_columns_to_check] <= 0).any(axis=1).sum()
print(f"Rows with zero or negative values: {invalid_rows:,}")

print(f"Clean dataset: {len(phones):,} phones ready for modelling.")

# Step 4: Feature Extraction

The model needs numbers, so a few raw columns are converted first. The function `build_features()` turns the raw phone columns into the **22 model inputs**:

| Raw column | Becomes | How |
|---|---|---|
| `resolution` (e.g. `1080x2400`) | `res_width`, `res_height` | Split the text at the `x` |
| `announcement_date` (e.g. `2022-03-01`) | `release_year` | year + (month − 1) / 12, so March 2022 = 2022.17 |
| `os` (e.g. `Android 12`) | `os_family`, `os_version` | Text before the number, and the first number |
| `video_*` (True / False) | `video_*` (1 / 0) | Boolean converted to a number |

The same function is used again in **Step 11.2** to convert a user's phone into a feature vector, so training and prediction always prepare the data in exactly the same way. Values that are fixed row by row (not learned from the data) can be created here without causing data leakage; anything learned from data (scaling, encoding, filling missing values) stays inside the pipeline in Step 5.

In [ ]:
video_columns = [
    "video_720p", "video_1080p", "video_4K", "video_8K",
    "video_30fps", "video_60fps", "video_120fps", "video_240fps",
    "video_480fps", "video_960fps",
]

numeric_features = (
    ["inches", "battery", "ram(GB)", "weight(g)", "storage(GB)",
     "res_width", "res_height", "release_year", "os_version"]
    + video_columns
)
categorical_features = ["brand", "os_family", "battery_type"]
target = "price(USD)"


def os_family_of(os_text):
    """'Android 12' -> 'Android', 'iOS 15.4' -> 'iOS', 'EMUI 12' -> 'EMUI', 'Tizen 3.0' -> 'Tizen'."""
    os_text = str(os_text).strip()
    for family in ["Android", "iOS", "EMUI", "Tizen"]:
        if os_text.startswith(family):
            return family
    return os_text


def build_features(raw):
    """Convert raw phone columns into the 22 model input columns."""
    features = pd.DataFrame(index=raw.index)

    # Numbers that are used as they are
    for column in ["inches", "battery", "ram(GB)", "weight(g)", "storage(GB)"]:
        features[column] = raw[column].astype(float)

    # '1080x2400' -> width 1080, height 2400
    size = raw["resolution"].str.extract(r"(\d+)\s*x\s*(\d+)").astype(float)
    features["res_width"] = size[0]
    features["res_height"] = size[1]

    # '2022-03-01' -> 2022 + 2/12
    date = pd.to_datetime(raw["announcement_date"])
    features["release_year"] = date.dt.year + (date.dt.month - 1) / 12

    # 'Android 12' -> family 'Android' and version 12 (version stays empty if the text has no number)
    features["os_family"] = raw["os"].map(os_family_of)
    features["os_version"] = pd.to_numeric(
        raw["os"].str.extract(r"(\d+(?:\.\d+)?)")[0], errors="coerce"
    )

    # True / False -> 1 / 0
    for column in video_columns:
        features[column] = raw[column].astype(int)

    features["brand"] = raw["brand"]
    features["battery_type"] = raw["battery_type"]

    return features[numeric_features + categorical_features]


# Input features (X) and output (y)
X = build_features(phones)
y = phones[target]

print(f"Input features (X): {X.shape[1]} columns -> {list(X.columns)}")
print(f"Output (y)        : {target}")
print(f"Number of phones  : {len(X):,}")
print(f"Average price     : ${y.mean():,.2f}")
print(f"Phones with no OS version number (filled in Step 5): {X['os_version'].isna().sum()}")

X.head()

# Step 5: Build the Pre-processing Pipeline

Machine learning algorithms need numbers, and linear algorithms (Ridge, Lasso) are sensitive to the scale of each feature. The pipeline follows the exact instructor architecture:

| Feature type | Step | Why |
|---|---|---|
| Numeric (inches, battery, ram, weight, storage, resolution, release year, OS version, video flags) | **Fill missing values** with median, then **Standard scaling** (mean 0, std 1) | Algorithms cannot handle empty values; scaling prevents large-range features (such as battery in mAh) from dominating small-range ones (such as screen inches) |
| Categorical (brand, os_family, battery_type) | **Fill missing values** with most frequent, then **One-hot encoding** | Turns text categories into 0/1 indicator columns |

### Why put pre-processing *inside* the pipeline?

In cross validation, fitting encoders or scalers on the **whole** dataset before splitting causes **data leakage**: the preprocessor would "see" the test fold.

By placing them inside the pipeline, the cross validation loop in Step 8 **re-fits them on the training folds only** in every round, and then applies them to the test fold. This is exactly what happens in the real world, where a model never sees future data during training.

`handle_unknown="ignore"` ensures that unseen categories in a test fold do not crash the pipeline.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

print("Pre-processing pipeline created:")
preprocessor

# Step 6: Define the 5 Regression Algorithms

We compare five algorithms, from regularised linear models to advanced ensemble methods:

| # | Algorithm | How it predicts a price |
|---|---|---|
| 1 | **Ridge Regression** | Fits a linear relationship with L2 regularisation to avoid overfitting |
| 2 | **Lasso Regression** | Fits a linear relationship with L1 regularisation (automatic feature selection: it can drop useless features) |
| 3 | **Extra Trees Regressor** | Builds 100 extremely randomized trees and averages their predictions |
| 4 | **AdaBoost Regressor** | Sequentially fits weak decision trees, focusing on harder examples |
| 5 | **XGBoost Regressor** | Scalable gradient-boosted trees with second-order gradients and regularization |

`random_state=42` makes all results reproducible across runs.

In [ ]:
algorithms = {
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    "Lasso Regression": Lasso(alpha=1.0, random_state=42, max_iter=2000),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100, min_samples_leaf=3, n_jobs=-1, random_state=42
    ),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=42),
    "XGBoost Regressor": XGBRegressor(
        n_estimators=350, learning_rate=0.07, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=42, verbosity=0, eval_metric="rmse"
    ),
}

print("Algorithms to compare:")
for number, name in enumerate(algorithms, start=1):
    print(f"  {number}. {name}")

# Step 7: Set Up 10-Fold Cross Validation
## Step 7.1: Create the K-Fold Splitter

- `n_splits=10`: split the data into 10 folds
- `shuffle=True`: shuffle the phones first so folds are representative. This matters here because the CSV is sorted by announcement date, so without shuffling the last fold would contain only the newest phones.
- `random_state=42`: the **same folds are used for all 5 algorithms**. Every algorithm is tested on exactly the same phones, which makes the comparison fair.

In [ ]:
K = 10
kfold = KFold(n_splits=K, shuffle=True, random_state=42)

print(f"{K}-Fold Cross Validation created (shuffle=True, random_state=42)")

## Step 7.2: Inspect the Folds

Let's check how many phones are used for training and testing in each round.

In [ ]:
fold_sizes = []
for fold_number, (train_index, test_index) in enumerate(kfold.split(X), start=1):
    fold_sizes.append({
        "Fold (Round)": fold_number,
        "Training Phones": len(train_index),
        "Testing Phones": len(test_index),
        "Avg Price of Test Phones ($)": y.iloc[test_index].mean(),
        "Most Expensive Test Phone ($)": y.iloc[test_index].max(),
    })

fold_sizes_table = pd.DataFrame(fold_sizes).set_index("Fold (Round)")
fold_sizes_table

Notice that the **most expensive test phone** is different in each fold. A fold that happens to contain several very expensive flagships will usually have a **higher RMSE**, because of the squaring explained in Step 3.1. With only 151 test phones per fold, this effect is stronger than it would be on a larger dataset, so expect the RMSE to vary noticeably from fold to fold.

# Step 8: Run 10-Fold Cross Validation for All 5 Algorithms

For each algorithm we build a pipeline (**pre-processing → algorithm**). Then, for each of the 10 folds, we:

1. make a **fresh copy** of the pipeline (`clone`), so nothing learned in one round is carried into the next,
2. **train** it on the training folds,
3. **predict** the prices of the phones in the test fold,
4. calculate the **RMSE** for that fold using `root_mean_squared_error`.

### Progress Bar (tqdm)

`tqdm` shows a **progress bar** while the loop runs. There is one step for each algorithm on each fold, so the bar has **5 × 10 = 50 steps**. While the cell runs, the bar looks like this:

```
XGBoost Regressor, fold 3/10:  44%|████████     | 22/50 [00:03<00:04,  6.10fold/s]
```

| Part of the bar | Meaning |
|---|---|
| `XGBoost Regressor, fold 3/10` | The algorithm and fold being trained right now |
| `44%` and `22/50` | How much of the work is done (steps done / total steps) |
| `00:03` (before `<`) | **Elapsed time**: time spent so far (minutes:seconds) |
| `00:04` (after `<`) | **Remaining time**: estimated time left (minutes:seconds) |
| `6.10fold/s` | Speed: folds per second (or `s/fold` when it is slow) |

> ⏱ **This step is quick (well under a minute in Colab)** because the dataset is small. Both XGBoost and Extra Trees use multi-threading (`n_jobs=-1`).

In [ ]:
fold_rmse = {}        # RMSE on each fold, for each algorithm
training_time = {}    # total time taken for all folds

folds = list(kfold.split(X))      # the same folds for every algorithm
total_steps = len(algorithms) * K

with tqdm(total=total_steps, unit="fold") as progress_bar:
    for name, algorithm in algorithms.items():
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", algorithm),
        ])

        scores = []
        start = time.time()

        for fold_number, (train_index, test_index) in enumerate(folds, start=1):
            progress_bar.set_description(f"{name}, fold {fold_number}/{K}")

            fold_model = clone(model)                         # fresh, untrained copy
            fold_model.fit(X.iloc[train_index], y.iloc[train_index])

            preds = fold_model.predict(X.iloc[test_index])
            rmse = root_mean_squared_error(y.iloc[test_index], preds)
            scores.append(rmse)

            progress_bar.update(1)

        fold_rmse[name] = scores
        training_time[name] = time.time() - start

# Step 9: Compare the Algorithms
## Step 9.1: RMSE of Each Algorithm on Each Fold

Each row is one round of cross validation, and each column is one algorithm. The **lowest RMSE in each row** (the best algorithm on that fold) is highlighted in green.

**How to read it:** count the green cells in each column to see how many folds each algorithm **won**. The algorithm with the best *average* does not always win every fold, which is why we compare the mean over all folds (Step 9.2) rather than trusting any single fold.

In [ ]:
per_fold_table = pd.DataFrame(fold_rmse)
per_fold_table.index = [f"Fold {i}" for i in range(1, K + 1)]
per_fold_table.index.name = "Fold"

(per_fold_table.style
    .format("${:,.0f}")
    .highlight_min(axis=1, props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))

## Step 9.2: Summary Comparison Table

This is the main result of the notebook. For each algorithm:

| Column | Meaning |
|---|---|
| **Mean RMSE** | Average RMSE over the 10 folds. This is the **main measure for ranking** the algorithms (lower is better) |
| **Std RMSE** | Standard deviation over the folds. A **small value means the algorithm is consistent**; a large value means its performance depends on which phones it gets |
| **Best / Worst Fold** | Lowest and highest RMSE across the folds |
| **RMSE as % of Avg Price** | Mean RMSE divided by the average phone price. This puts the error in context |
| **Time** | Total time to train and test across all 10 folds |
| **Rank** | 1 = lowest mean RMSE = best algorithm |

In [ ]:
average_price = y.mean()

summary = pd.DataFrame({
    "Mean RMSE ($)": per_fold_table.mean(),
    "Std RMSE ($)": per_fold_table.std(),
    "Best Fold RMSE ($)": per_fold_table.min(),
    "Worst Fold RMSE ($)": per_fold_table.max(),
    "RMSE as % of Avg Price": per_fold_table.mean() / average_price * 100,
    "Time (s)": pd.Series(training_time),
})
summary = summary.sort_values("Mean RMSE ($)")
summary.insert(0, "Rank", range(1, len(summary) + 1))
summary.index.name = "Algorithm"

print(f"Average phone price in this data: ${average_price:,.2f}\n")

money_columns = ["Mean RMSE ($)", "Std RMSE ($)", "Best Fold RMSE ($)", "Worst Fold RMSE ($)"]
(summary.style
    .format({**{c: "${:,.0f}" for c in money_columns},
             "RMSE as % of Avg Price": "{:.1f}%",
             "Time (s)": "{:.1f}s"})
    .highlight_min(subset=["Mean RMSE ($)"],
                   props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;")
    .highlight_max(subset=["Mean RMSE ($)"],
                   props="background-color: #f9d0d0; color: #0b0b0b;"))

## Step 9.3: Visual Comparison

**Chart 1** shows the mean RMSE of each algorithm (shorter bar = better). The black whisker shows ± 1 standard deviation across the 10 folds.

**Chart 2** shows the RMSE of **every individual fold** as a dot, so you can see how spread out each algorithm's results are.

In [ ]:
# Chart colours — matching the instructor's notebook palette
BAR_COLOR = "#2a78d6"
TEXT_COLOR = "#0b0b0b"
MUTED_TEXT = "#52514e"
GRID_COLOR = "#e4e3df"
dollars = FuncFormatter(lambda value, _: f"${value:,.0f}")

def style_axis(ax):
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.tick_params(colors=MUTED_TEXT, length=0)
    ax.xaxis.set_major_formatter(dollars)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)

order = summary.index.tolist()            # best algorithm first
positions = np.arange(len(order))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2))

# ---- Chart 1: mean RMSE with +/- 1 std ----
means = summary["Mean RMSE ($)"].values
stds = summary["Std RMSE ($)"].values

ax1.barh(positions, means, color=BAR_COLOR, height=0.55, zorder=2)
ax1.errorbar(means, positions, xerr=stds, fmt="none",
             color=TEXT_COLOR, capsize=4, linewidth=1.5, zorder=3)

for pos, mean, std in zip(positions, means, stds):
    ax1.text(mean + std + stds.max() * 0.08, pos, f"${mean:,.0f}",
             va="center", ha="left", color=TEXT_COLOR, fontsize=9)

ax1.set_yticks(positions)
ax1.set_yticklabels(order, color=TEXT_COLOR, fontsize=10)
ax1.invert_yaxis()
ax1.set_xlabel("RMSE (USD)", color=MUTED_TEXT)
ax1.set_title("Mean RMSE (± 1 Std Dev) per Algorithm", color=TEXT_COLOR, fontweight="bold")
style_axis(ax1)

# ---- Chart 2: per-fold RMSE dots ----
for i, name in enumerate(order):
    fold_values = per_fold_table[name].values
    ax2.scatter(fold_values, [i] * K, color=BAR_COLOR, s=40, zorder=2, alpha=0.7)
    ax2.plot([fold_values.min(), fold_values.max()], [i, i],
             color=GRID_COLOR, linewidth=1, zorder=1)

ax2.set_yticks(positions)
ax2.set_yticklabels(order, color=TEXT_COLOR, fontsize=10)
ax2.invert_yaxis()
ax2.set_xlabel("RMSE (USD)", color=MUTED_TEXT)
ax2.set_title("Per-Fold RMSE (each dot = one fold)", color=TEXT_COLOR, fontweight="bold")
style_axis(ax2)

fig.suptitle("Smartphone Price Prediction — Algorithm Comparison (10-Fold CV)",
             color=TEXT_COLOR, fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Step 9.4: Observations

> The numbers below reflect 10-fold cross validation on all 1,510 phones (after cleaning).

1. **Extra Trees achieves the lowest mean RMSE (~$225)**, followed by Lasso (~$233), Ridge (~$234) and XGBoost (~$237). Extra Trees also had the lowest RMSE on 5 of the 10 folds (Lasso and Ridge won 2 folds each, XGBoost won 1).
2. **The top four algorithms are close.** The gap between the best (~$225) and the fourth (~$237) is only about $12, while the fold-to-fold standard deviation is about $30–$36. So the ranking among these four should be read as "similar performance, Extra Trees slightly ahead", not as a large win.
3. **Linear models are competitive on this dataset.** Unlike a dataset with very strong non-linear effects, phone prices rise fairly steadily with RAM, storage, battery, screen quality and release date, so the regularised linear models (Ridge and Lasso) get close to the tree models. Their results are almost identical (~$233 and ~$234): both fit a linear relationship on the same scaled inputs, so Lasso's extra feature selection brings only a very small difference.
4. **XGBoost did not win here.** With only 1,359 training phones per round, a boosted model of 350 trees with depth 7 has little data to learn from, which probably lets it fit noise in the training folds; Extra Trees averages many randomized trees with a minimum leaf size of 3, which is steadier on small data. This is the opposite of what happens on very large datasets, where boosting usually wins.
5. **AdaBoost performs worst (~$316)**, roughly $90 worse than the best model. Its default setting (50 very shallow trees) is too simple to capture the interactions between brand, memory and screen specifications.
6. **Variation across folds is large.** The standard deviation is about $30–$36 and the best Extra Trees fold (~$176) is far below its worst fold (~$290). With only 151 test phones per fold, a few expensive flagships in one fold change its RMSE noticeably (see Step 3.1 and Step 7.2). This is the reason we trust the 10-fold average rather than a single split.
7. **Practical error rate:** the best RMSE is about 67% of the average phone price ($338). This is high, which shows that specifications alone do not fully explain price: brand value and market positioning also matter and are not in the data. Because RMSE is dominated by the expensive phones, the error on typical budget and mid-range phones is a smaller dollar amount than this figure suggests.

# Step 10: Train the Final Model
## Step 10.1: Select the Best Algorithm

**Cross validation is used to *choose* an algorithm, not to produce the final model.** The 10 models built during cross validation were only used to measure performance and are now discarded.

We select the algorithm with the **lowest mean RMSE** from the summary table.

In [ ]:
best_algorithm_name = summary.index[0]
best_mean_rmse = summary.loc[best_algorithm_name, "Mean RMSE ($)"]
best_std_rmse = summary.loc[best_algorithm_name, "Std RMSE ($)"]

print(f"Best algorithm : {best_algorithm_name}")
print(f"Expected RMSE  : ${best_mean_rmse:,.2f} (+/- ${best_std_rmse:,.2f})")

## Step 10.2: Train the Best Algorithm on All the Data

Now that we know which algorithm works best, we train it on **all 1,510 phones**. There is no separate test set here, because the cross-validation RMSE from Step 9 is already our unbiased estimate of how well this model will perform on new, unseen phones.

In [ ]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", algorithms[best_algorithm_name]),
])

final_model.fit(X, y)

print(f"Final {best_algorithm_name} model trained on {len(X):,} phones.")

## Step 10.3: Save the Trained Model

The saved file contains the **whole pipeline** (imputer, scaler, one-hot encoder **and** the algorithm), so it can predict directly from the 22 feature columns without any manual preprocessing.

To download `model.pkl` to your computer in Colab, set `DOWNLOAD_MODEL = True` and run the cell again, or use the **Files** panel on the left.

> **Important for deployment:** `model.pkl` must be committed to your GitHub repository so the Streamlit app can load it. See `README.md` for instructions.

In [ ]:
MODEL_FILE = "model.pkl"
DOWNLOAD_MODEL = False

import joblib

# Save with joblib compression (compress=3) to keep the file small
joblib.dump(final_model, MODEL_FILE, compress=3)

size_bytes = os.path.getsize(MODEL_FILE)
size_mb = size_bytes / (1024 * 1024)

print(f"Trained model saved as '{MODEL_FILE}'")
print(f"File size: {size_mb:.2f} MB")

if size_mb > 90:
    print("WARNING: model exceeds 90 MB limit.")
else:
    print("✓ File size is within the 90 MB limit.")

if IN_COLAB and DOWNLOAD_MODEL:
    files.download(MODEL_FILE)

# Step 11: Execute the Application Phase
## Step 11.1: Take Input from User

By default the notebook uses the example phone below, so that **Run all** works without stopping. To type in your own phone, set `USE_INTERACTIVE_INPUT = True` and run the cell again.

In [ ]:
USE_INTERACTIVE_INPUT = False

# Default example: a Samsung phone running Android 12, announced in March 2022
example_phone = {
    "brand": "Samsung",
    "os": "Android 12",
    "inches": 6.5,
    "resolution": "1080x2400",
    "battery": 5000,
    "battery_type": "Li-Po",
    "ram(GB)": 8,
    "announcement_date": "2022-03-01",
    "weight(g)": 195.0,
    "storage(GB)": 128,
    "video_720p": True,
    "video_1080p": True,
    "video_4K": True,
    "video_8K": False,
    "video_30fps": True,
    "video_60fps": True,
    "video_120fps": False,
    "video_240fps": False,
    "video_480fps": False,
    "video_960fps": False,
}

def ask_yes_no(question):
    return input(f"  {question} (y/n): ").strip().lower().startswith("y")

def get_phone_input():
    print("Enter Phone Specifications (valid ranges from dataset shown):")
    print(f"  brand        : {', '.join(sorted(X['brand'].unique()))}")
    print("  os           : e.g. Android 12, iOS 15, EMUI 12")
    print(f"  inches       : {X['inches'].min():.1f} - {X['inches'].max():.1f}")
    print("  resolution   : e.g. 1080x2400")
    print(f"  battery (mAh): {X['battery'].min():,.0f} - {X['battery'].max():,.0f}")
    print("  battery_type : Li-Po, Li-Ion")
    print(f"  ram (GB)     : {X['ram(GB)'].min():.0f} - {X['ram(GB)'].max():.0f}")
    print("  announcement : YYYY-MM-DD (2016-09-01 to 2023-08-31 in the dataset)")
    print(f"  weight (g)   : {X['weight(g)'].min():.0f} - {X['weight(g)'].max():.0f}")
    print(f"  storage (GB) : {X['storage(GB)'].min():.0f} - {X['storage(GB)'].max():.0f}")
    phone = {
        "brand": input("  Enter brand        : ").strip(),
        "os": input("  Enter os           : ").strip(),
        "inches": float(input("  Enter inches       : ")),
        "resolution": input("  Enter resolution   : ").strip(),
        "battery": int(input("  Enter battery (mAh): ")),
        "battery_type": input("  Enter battery_type : ").strip(),
        "ram(GB)": int(input("  Enter ram (GB)     : ")),
        "announcement_date": input("  Enter announcement : ").strip(),
        "weight(g)": float(input("  Enter weight (g)   : ")),
        "storage(GB)": int(input("  Enter storage (GB) : ")),
    }
    print("Video recording support:")
    for column in video_columns:
        phone[column] = ask_yes_no(f"Supports {column.replace('video_', '')} video?")
    return phone

if USE_INTERACTIVE_INPUT:
    user_input = get_phone_input()
else:
    user_input = example_phone
    print("Using example phone:")
    for k, v in user_input.items():
        print(f"  {k:18s}: {v}")

## Step 11.2: Convert User Input into Feature Vector

The feature vector must have **exactly the same columns, in the same order,** as the training data. We reuse `build_features()` from Step 4, so the resolution, date and OS text are converted in exactly the same way as during training. There is no need to encode or scale it manually: the saved pipeline handles everything automatically.

In [ ]:
feature_vector = build_features(pd.DataFrame([user_input]))
feature_vector

## Step 11.3: Load the Saved Model

In [ ]:
import joblib

try:
    loaded_model = joblib.load(MODEL_FILE)
except Exception:
    with open(MODEL_FILE, "rb") as f:
        loaded_model = pickle.load(f)

print(f"Model loaded from '{MODEL_FILE}': {loaded_model.named_steps['regressor'].__class__.__name__}")

## Step 11.4: Model Prediction

Apply the model to the feature vector of the unseen phone and return the prediction to the user.

In [ ]:
predicted_price = loaded_model.predict(feature_vector)[0]

print(f"Predicted Price: ${predicted_price:,.2f} USD")
print(f"Typical error (cross-validation RMSE of {best_algorithm_name}): about ${best_mean_rmse:,.0f}")

# Step 12: Execute the Feedback Phase

After deployment, feedback from users and domain experts (phone retailers, resellers, market analysts) is used to improve the model. Potential improvements for this system:

1. **Hyperparameter Tuning**: optimize tree depth, number of trees, learning rate and regularisation using `RandomizedSearchCV` or Bayesian optimisation across folds. The XGBoost settings in this notebook were chosen for a large dataset and may be too complex for 1,510 phones.
2. **Target Transformation**: evaluate a `log(price)` transformation. Prices are right-skewed, so predicting the log price would penalise percentage errors equally on budget phones and flagships instead of letting expensive phones dominate the RMSE.
3. **Feature Engineering**: add features such as pixel density (resolution divided by screen size), the age of the phone at prediction time, or a brand-tier indicator to capture the brand premium that the current features miss.
4. **More Data**: add more phones (especially flagships and recent releases) and extra specifications such as chipset, camera megapixels and display type, which strongly influence real-world pricing.
5. **Deployment Monitoring**: log user queries and real-world prices to detect drift, because phone prices fall as models age and new generations are released.
6. **App Enhancements**: add batch upload (CSV) so shops can price complete inventory lists in one click.

# Conclusion

In this notebook we compared **5 regression algorithms** on the **full smartphone dataset (1,510 phones after cleaning)** using **10-fold cross validation** and **RMSE**.

- **Extra Trees** achieved the lowest mean RMSE (~$225), narrowly ahead of **Lasso** (~$233), **Ridge** (~$234) and **XGBoost** (~$237). Extra Trees was selected, retrained on all phones, saved as `model.pkl`, and verified inside the application phase.
- **AdaBoost performed worst (~$316)**, because its default shallow trees are too simple for the interactions between brand, memory and screen specifications.
- The top four algorithms are within about $12 of each other, which is smaller than the fold-to-fold standard deviation (~$30–$36). The ranking among them is therefore a small edge, not a large difference. On a small dataset of about 1,500 phones, the simple regularised linear models stay competitive with boosted trees.
- The 10-fold cross validation setup guarantees that every phone is used for testing exactly once, giving a more dependable estimate than a single train/test split. The best model's error (~67% of the average price) shows that specifications alone do not explain phone prices completely; brand value and market positioning also matter.

### Algorithm Ranking Summary

| Rank | Algorithm | Mean RMSE ($) | Std RMSE ($) |
|:---:|---|:---:|:---:|
| 1 | Extra Trees | ~$225 | ~$33 |
| 2 | Lasso Regression | ~$233 | ~$30 |
| 3 | Ridge Regression | ~$234 | ~$30 |
| 4 | XGBoost Regressor | ~$237 | ~$36 |
| 5 | AdaBoost Regressor | ~$316 | ~$32 |